In [4]:
import os
import subprocess as sp
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

In [12]:
model = init_chat_model(
    model = "gemini-3.1-flash-lite",
    model_provider="google_genai"
    )
model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.8.1', 'langchain': '1.5.0', 'langchain-google-genai': '4.5.0'}}, profile={'name': 'Gemini 3.1 Flash Lite', 'release_date': '2026-05-07', 'last_updated': '2026-05-07', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), model='gemini-3.1-flash-lite', temperature=None, client=<google.genai.client.Client object at 0x11c3f1250>, default_metadata=(), model_kwargs={})

In [13]:
from langchain.tools import tool

@tool
def get_date()->str:
    """Get the current local system date."""
    return sp.run(["date"],capture_output=True, text=True)


@tool
def get_all_process() -> str:
    """List processes associated with terminals across all users."""
    result = sp.run(["ps", "-a"], capture_output=True, text=True)
    return result.stdout


@tool
def get_process() -> str:
    """List processes associated with the current user's terminal."""
    result = sp.run(["ps"], capture_output=True, text=True)
    return result.stdout

In [15]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[get_date, get_all_process, get_process],
    system_prompt="You are a helpful assistant"
)

In [17]:
response = agent.invoke({"messages":[{"role":"user", "content":"What are the processes associated with my termainl"}]})
response["messages"][-1].content

[{'type': 'text',
  'text': 'The processes currently associated with your terminals are:\n\n*   **PID 2653:** `/bin/zsh -i` (on `ttys002`)\n*   **PID 3003:** `/bin/zsh -il` (on `ttys003`)\n*   **PID 12731:** `/bin/zsh -il` (on `ttys004`)',
  'extras': {'signature': 'EpwBCpkBAWkUfRN3MMtDO616J74PeHy8YfbCSGJiWZMRPsmVJAayVYeXItMAWKrTRjFowHM2nRGMJBAYD0ErviRLyvzrmEJ+wUIgA0pymNx1RIC4VIXlJKXMwQ+coQWU2tocTMjzeqgEK46M8h4dJ/C1G36XOVkvuXyP+Qiakcuhtr9q+5c48cGDLTQemYw5k54pDa2Jb65lsVroBNeF'}}]